<div style="text-align: center; font-family: 'Georgia', serif; padding: 30px 60px; border-bottom: 2px solid #333; margin-bottom: 20px;">

<h1 style="font-size: 1.6em; font-weight: bold; margin-bottom: 8px;">
Dataset Preprocessing: Tic-Tac-Toe Endgame<br>Standardization and Feature Engineering
</h1>

<p style="font-size: 1.0em; color: #444; margin: 6px 0;">
<strong>Júlia Augustini, Pedro Tomielo, Rodrigo Gonçalves</strong>
</p>

<p style="font-size: 0.9em; color: #555; margin: 4px 0;">
Pontifícia Universidade Católica — Cadeira: Inteligência Artificial
</p>

<p style="font-size: 0.85em; color: #666; margin: 4px 0;">
Abril de 2026
</p>

</div>

---

**Resumo.** Este notebook aplica preprocessamento e padronização ao dataset Tic-Tac-Toe Endgame. As operações incluem padronização de valores (x→1, o→-1, b→0), detecção de duplicatas, geração de casos espelhados de empate para obter representação completa (32 empates) e criação da coluna de resultado com classificação do estado do jogo.

In [146]:
import pandas as pd
import numpy as np

# Carregar o dataset original
df = pd.read_csv("../../data/processed/eda_1.csv")
df.head()


,pos_1,pos_2,pos_3,pos_4,pos_5,pos_6,pos_7,pos_8,pos_9,classe
0,x,x,x,x,o,o,x,o,o,X venceu
1,x,x,x,x,o,o,o,x,o,X venceu
2,x,x,x,x,o,o,o,o,x,X venceu
3,x,x,x,x,o,o,o,b,b,X venceu
4,x,x,x,x,o,o,b,o,b,X venceu


## Padronização de Valores

Transformar valores categóricos em numéricos:
- 'x' → 1 (jogador X)
- 'o' → -1 (jogador O)
- 'b' → 0 (célula vazia)

In [147]:
import pandas as pd
import numpy as np
import random

# Definindo as combinações de vitória (linhas, colunas, diagonais)
WIN_LINES = [
    [0, 1, 2], [3, 4, 5], [6, 7, 8],  # Linhas
    [0, 3, 6], [1, 4, 7], [2, 5, 8],  # Colunas
    [0, 4, 8], [2, 4, 6]              # Diagonais
]

def classificar_vitoria(board):
    """
    Verifica se há um vencedor no tabuleiro numérico.
    Soma 3 significa vitória do X, Soma -3 significa vitória do O.
    """
    for line in WIN_LINES:
        line_sum = sum(board[i] for i in line)
        if line_sum == 3:
            return 1  # X venceu
        if line_sum == -3:
            return -1 # O venceu
    return 0 # Sem vitória (Tem jogo ou Empate)

# --- PASSO 1: Padronização Numérica ---
def padronizar_dataset(df):
    """Transforma 'x' em 1, 'o' em -1 e 'b' em 0."""
    print("Passo 1: Padronizando o dataset...")
    mapa = {'x': 1, 'o': -1, 'b': 0}
    df_padrao = df.copy()
    colunas_pos = [f'pos_{i}' for i in range(1, 10)]
    
    for col in colunas_pos:
        df_padrao[col] = df_padrao[col].map(mapa).fillna(df_padrao[col])
    
    return df_padrao

# --- PASSO 2: Expansão dos Casos de Empate ---
def expandir_empates(df):
    """Gera o espelhamento das configurações de empate (x vira o, o vira x)."""
    print("Passo 2: Expandindo casos de empate...")
    colunas_pos = [f'pos_{i}' for i in range(1, 10)]
    
    empates = df[df['classe'] == 'Empate'].copy()
    empates_espelhados = empates.copy()
    
    # Invertendo os sinais (1 vira -1, e -1 vira 1. O zero continua zero)
    for col in colunas_pos:
        empates_espelhados[col] = empates_espelhados[col] * -1
        
    df_expandido = pd.concat([df, empates_espelhados], ignore_index=True)
    df_expandido = df_expandido.drop_duplicates(subset=colunas_pos)
    return df_expandido

# --- PASSOS 3 a 5: Geração Aleatória de Estados Intermediários ---
def gerar_estados_intermediarios(df, n_geracao_por_linha=3, seed=42):
    """Remove 1 a 3 posições preenchidas e cria a classe 'Tem jogo'."""
    print(f"Passos 3, 4 e 5: Gerando estados intermediários (n={n_geracao_por_linha})...")
    np.random.seed(seed)
    random.seed(seed)
    
    colunas_pos = [f'pos_{i}' for i in range(1, 10)]
    novos_estados = []
    
    for _, row in df.iterrows():
        board = row[colunas_pos].values.astype(int)
        
        variacoes_criadas = 0
        tentativas = 0
        
        # Limite de tentativas para não entrar em loop infinito se não houver soluções
        while variacoes_criadas < n_geracao_por_linha and tentativas < 20:
            tentativas += 1
            novo_board = board.copy()
            
            # Índices de posições preenchidas (1 ou -1)
            pos_preenchidas = np.where(novo_board != 0)[0]
            
            if len(pos_preenchidas) == 0:
                break
                
            # Remove de 1 a 3 posições
            num_remover = random.randint(1, min(3, len(pos_preenchidas)))
            pos_remover = random.sample(list(pos_preenchidas), num_remover)
            
            for p in pos_remover:
                novo_board[p] = 0
                
            # Validação 1: Tem posição vazia? (Garantido pela remoção, mas podemos checar)
            # Validação 2: Tem vitória? (check_vitoria deve retornar 0)
            # Validação Extra (Opcional, mas recomendada): 
            # Num jogo válido de jogo da velha, X sempre começa. 
            # Logo a soma do board deve ser 0 (peças iguais) ou 1 (X tem uma a mais).
            if classificar_vitoria(novo_board) == 0 and (0 <= sum(novo_board) <= 1):
                novo_estado = {f'pos_{i+1}': novo_board[i] for i in range(9)}
                novo_estado['classe'] = 'Tem jogo'
                novos_estados.append(novo_estado)
                variacoes_criadas += 1

    df_novos = pd.DataFrame(novos_estados)
    # Remove eventuais duplicatas geradas pela aleatoriedade
    df_novos = df_novos.drop_duplicates(subset=colunas_pos)
    return df_novos

# --- PASSO 6: Combinando Tudo (Pipeline Final) ---
# --- PASSO 6: Combinando Tudo (Pipeline Final Corrigido) ---
def pipeline_data_augmentation(df):
    """Executa todas as etapas e retorna o dataset final balanceado."""
    
    # 1. Padroniza ('x', 'o', 'b' -> 1, -1, 0)
    df_numerico = padronizar_dataset(df)
    
    # 2. Expande empates (16 -> 32)
    df_com_empates = expandir_empates(df_numerico)
    
    # 3. Gera 'Tem jogo' (n=2 para garantir que gere um pouco mais de 626 antes de cortar)
    df_intermediarios = gerar_estados_intermediarios(df_com_empates, n_geracao_por_linha=2)
    
    # CORREÇÃO: Pega o tamanho da classe majoritária automaticamente (sem depender do texto exato)
    # Supondo que a sua coluna se chame 'classe' (se for outro nome, mude aqui embaixo)
    nome_coluna_alvo = 'classe' 
    alvo_amostras = df[nome_coluna_alvo].value_counts().max() # Retornará 626
    
    # Limita o tamanho do 'Tem jogo' para não desbalancear
    if len(df_intermediarios) > alvo_amostras:
        df_intermediarios = df_intermediarios.sample(n=alvo_amostras, random_state=42)
    
    # 4. Combina tudo
    print("Passo 6: Combinando tudo...")
    df_final = pd.concat([df_com_empates, df_intermediarios], ignore_index=True)
    
    # Remover duplicatas globais
    colunas_pos = [f'pos_{i}' for i in range(1, 10)]
    df_final = df_final.drop_duplicates(subset=colunas_pos, keep='first')
    
    print("\nProcesso concluído! Distribuição final das classes:")
    print(df_final[nome_coluna_alvo].value_counts())
    
    return df_final

In [148]:
df_aumentado = pipeline_data_augmentation(df)
df_aumentado


Passo 1: Padronizando o dataset...
Passo 2: Expandindo casos de empate...
Passos 3, 4 e 5: Gerando estados intermediários (n=2)...
Passo 6: Combinando tudo...

Processo concluído! Distribuição final das classes:
classe
X venceu    942
Tem jogo    942
Empate       32
Name: count, dtype: int64


,pos_1,pos_2,pos_3,pos_4,pos_5,pos_6,pos_7,pos_8,pos_9,classe
0,1,1,1,1,-1,-1,1,-1,-1,X venceu
1,1,1,1,1,-1,-1,-1,1,-1,X venceu
2,1,1,1,1,-1,-1,-1,-1,1,X venceu
3,1,1,1,1,-1,-1,-1,0,0,X venceu
4,1,1,1,1,-1,-1,0,-1,0,X venceu
...,...,...,...,...,...,...,...,...,...,...
1911,1,1,0,0,0,-1,-1,1,-1,Tem jogo
1912,0,0,-1,1,1,0,0,1,-1,Tem jogo
1913,0,1,0,0,1,-1,1,-1,0,Tem jogo
1914,0,0,-1,0,1,-1,0,1,0,Tem jogo


In [149]:
# Salvando para a próxima etapa (Passo 7)
df_aumentado.to_csv('../../data/processed/preprocessed_1.csv', index=False)